# 04: Volt-VAr conformance, three metrics

Pick a **source table**, a **metric**, a **tolerance** and a **failure definition** in the parameters cell, then run all.

| Metric | Question | Counts |
|---|---|---|
| `nonconformance_frequency` | How often is the inverter out of conformance? | failures in all zones / all valid intervals; shown with voltage exposure |
| `remote_conformance_rate` (default) | Does the inverter respond when it is asked to? | failures / (zone A intervals + zone C failures) |
| `legacy_conformance_rate` | The original Stage 2 headline (back-comparison) | P >= 20% at any voltage, nearest-edge categories |

Zones: **A: response required** (P >= 20%, V < 220 or > 240 V) · **B: sufficient power, voltage deadband** (P >= 20%, 220-240 V) · **C: low power, voltage exposed** (P < 20%, V < 220 or > 240 V) · **D: low power, voltage deadband** (P < 20%, 220-240 V).

Site tag: conformant if the non-conforming share is at most 10%, per metric.

**Cost.** Sections 1-4 read only the small Stage 2 conformance tables. Section 5 plots sites from the local Solar Analytics store; it reads Athena telemetry only if `ALLOW_ATHENA_SITE_LOADS = True` (a few GB per site-day).

**Before committing:** outputs are stripped by the repository's notebook filter; never paste site-level results into markdown cells.

In [ ]:
%matplotlib inline

import importlib
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd().resolve()
REPO_ROOT = next(p for p in (HERE, *HERE.parents) if (p / "bms_sa_review").is_dir())
PACKAGE_ROOT = REPO_ROOT / "bms_sa_review"
for path in (REPO_ROOT, PACKAGE_ROOT, PACKAGE_ROOT / "shared", PACKAGE_ROOT / "data_query" / "lib"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import analysis_contract as contract
import conformance_metrics as cm
import conformance_plots as cp
import conformance_queries as cq
import vvar_results as vr
from bms_sa_review.shared.aws_config import aq
from standard_plots.lib import g_metric_compare as G
from standard_plots.lib import palette as PAL

# Reload the analysis modules after code changes (restart the kernel if
# shared/ciccada_config.py changed).
for module in (contract, cq, cm, cp, vr, G):
    importlib.reload(module)

PAL.use_style("notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## Parameters

In [ ]:
# ---- What to compute -------------------------------------------------------
SOURCE = "conformance_voltvar_zones"     # "conformance_voltvar_zones" (all three metrics, tol 0/4%)
                                         # "conformance_voltvar" (original table: legacy rate only, tol 4%)
METRIC = "remote_conformance_rate"       # "remote_conformance_rate" | "nonconformance_frequency" | "legacy_conformance_rate"
TOL = 0.04                               # 0.04 (default) or 0.0 (literal reading); zone table only
FAILURE_DEFINITION = "reduced"           # "reduced" (default) or "strict" (+ over-response)

YEARS = (2024, 2025)
DAY_NIGHT = "all"                        # "all" (the metrics are defined over all timestamps), "day", "night"

# ---- Breakdowns -------------------------------------------------------------
GROUPS = ("state", "dnsp", "oem", "install_year")
MIN_SITES = 20                           # hide groups with fewer sites

# ---- Sites ------------------------------------------------------------------
COMPARE = ("legacy_conformance_rate", "remote_conformance_rate")   # which tag flips to list
N_EXAMPLES = 2                           # example sites per change class (section 5)
SITE_IDS = []                            # or your own list of site_id to plot
ALLOW_ATHENA_SITE_LOADS = False          # True: sites not in the local store are read from Athena (costs)

CONFIG = contract.AnalysisConfig(
    years=YEARS,
    flex_selection="exclude",            # metadata matches the zone table (flex-export sites removed)
    day_night=DAY_NIGHT,
).validate()

LABEL = vr.describe(CONFIG, SOURCE, METRIC, TOL, FAILURE_DEFINITION)
print(LABEL)
display(contract.manifest(CONFIG))

## 1. Fleet result

In [ ]:
sites, all_metrics = vr.site_results(
    aq, CONFIG, source=SOURCE, metric=METRIC, tol=TOL, failure_definition=FAILURE_DEFINITION
)
meta = cq.fetch_metadata(aq, CONFIG).drop_duplicates("site_id")

result = cm.fleet_result(sites, METRIC, LABEL, "see the metric table above")
result["n_not_assessable"] = sites.attrs.get("n_not_assessable", 0)
display(result)

if all_metrics is not None:
    # All three metrics on the same sites, same tolerance, for orientation.
    overview = cm.vvar_metric_overview(all_metrics)
    display(overview)
    cp.plot_fleet_summary(overview.rename(columns={"sites_conformant_pct": "fleet_conformant_pct"}))
    if METRIC == "nonconformance_frequency":
        print(f"median voltage exposure {100 * sites['voltage_exposure'].median():.1f}% · "
              f"median coverage {100 * sites['coverage'].median():.1f}%")
cp.plot_site_distributions([(METRIC, sites)]);

## 2. Breakdowns by state, DNSP, manufacturer, installation year

In [ ]:
breakdowns = {}
for group in GROUPS:
    breakdowns[group] = cm.group_breakdown(sites, meta, group, min_sites=MIN_SITES)
    print(f"\n{METRIC} by {group}")
    display(breakdowns[group])
    cp.plot_group_breakdown(breakdowns[group], group, f"{METRIC} by {group} (tol {TOL:.0%})")

## 3. Why the remote conformance rate tags fewer sites conformant than the legacy rate

One change at a time, on the same sites (assessable under both metrics). The legacy rate counts every interval with P >= 20% at any voltage, so zone B (sufficient power, voltage deadband) intervals, which the legacy rule never fails, sit in its denominator and dilute the failures. The remote rate counts only intervals where a response was required (zone A) plus low-power failures where the voltage was outside the deadband (zone C). Zone B and zone D failures (voltage deadband violations) do not enter the remote rate at all.

In [ ]:
if all_metrics is None:
    print("Needs the zone table (SOURCE = 'conformance_voltvar_zones').")
else:
    steps = cm.vvar_legacy_to_remote_steps(all_metrics)
    display(steps)
    print("median share of the legacy denominator that is zone B (voltage deadband): "
          f"{steps.attrs['zone_B_share_of_legacy_denominator_median_pct']:.1f}%")
    cp.plot_group_breakdown(
        steps.rename(columns={"sites_conformant_pct": "fleet_conformant_pct"}),
        "step", f"From legacy to remote, one change at a time (tol {TOL:.0%})",
    )

## 4. Conformant and non-conformant sites, and sites whose tag flips

In [ ]:
cols = ["site_id", "verdict", "nonconf_frac", "voltage_exposure", "coverage", "n_valid", "n_A"]
tags = sites[[c for c in cols if c in sites.columns]].merge(
    meta[["site_id", "state", "dnsp", "oem", "install_year"]], on="site_id", how="left")
display(tags["verdict"].value_counts())

conformant_sites = tags[tags.verdict == "conformant"].sort_values("n_A", ascending=False)
nonconformant_sites = tags[tags.verdict == "non-conformant"].sort_values("nonconf_frac", ascending=False)
display(conformant_sites.head(10))
display(nonconformant_sites.head(10))

In [ ]:
# Sites whose tag changes between the two COMPARE metrics (same tolerance).
if all_metrics is None:
    print("Needs the zone table (SOURCE = 'conformance_voltvar_zones').")
else:
    a, b = COMPARE
    flips = G.flip_table(
        all_metrics.merge(meta[["site_id", "state", "dnsp", "install_year"]], on="site_id", how="left"),
        a, b, key="site_id", extra=("state", "dnsp", "install_year"),
    )
    display(flips["change"].value_counts().rename("sites").to_frame())
    display(flips.groupby("change")[[c for c in flips.columns if c.startswith("failures_in_zone")]]
            .median().round(1))
    examples = G.pick_examples(flips, n_changed=N_EXAMPLES, n_same=1)
    display(examples)

## 5. Plot sites

For each site (examples from section 4, or `SITE_IDS`): the day with the most zone A (response required) intervals, one column per metric (the band is the same; only what is counted changes), then the Q-V plot of that month. Telemetry comes from the local Solar Analytics store; sites not in it are skipped unless `ALLOW_ATHENA_SITE_LOADS = True`.

In [ ]:
to_plot = list(SITE_IDS) or (list(examples["site_id"]) if all_metrics is not None else [])

for site_id in to_plot:
    date = vr.best_day(aq, CONFIG, site_id, TOL)
    try:
        df, info = vr.site_day_frame(site_id, date, ALLOW_ATHENA_SITE_LOADS)
        G.plot_metric_comparison_day(df, info, tol=TOL, failure_definition=FAILURE_DEFINITION)
        dfm, infom = vr.site_month_frame(site_id, date[:7], ALLOW_ATHENA_SITE_LOADS)
        G.plot_qv_metric_comparison(dfm, infom, tol=TOL, failure_definition=FAILURE_DEFINITION)
    except LookupError as exc:
        print(f"site {site_id}: skipped. {exc}")